<a href="https://colab.research.google.com/github/royalgarter/cofog-panel/blob/main/cofog.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# COFOG-Panel: Harmonizing IMF GFS COFOG Data into Reproducible Panel Datasets

This notebook automates the setup, installation, and execution of the COFOG-Panel pipeline based on the project [README](README.md).

In [1]:
# @title 1. Setup Environment and Install Dependencies
# @markdown This cell installs dependencies and sets up the package in editable mode.

!git clone https://github.com/royalgarter/cofog-panel.git
%cd cofog-panel

# Install dependencies from requirements.txt
!pip install -r requirements.txt

# Install the package in editable mode to use the `cofog-panel` CLI command
!pip install -e .

Cloning into 'cofog-panel'...
remote: Enumerating objects: 237, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 237 (delta 10), reused 1 (delta 0), pack-reused 215 (from 2)
Receiving objects: 100% (237/237), 51.19 MiB | 14.96 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/cofog-panel
Obtaining file:///content/cofog-panel
  Preparing metadata (setup.py) ... done
  Running setup.py develop for cofog-panel


## 2.1. Prepare Data

Ensure your input files are placed in the `./data/` directory:
- `gfs_raw_data.xlsx` (Main COFOG source)
- `country_codes.xlsx` (Country mapping file)

In [2]:
import os

# Create necessary directories
os.makedirs('data', exist_ok=True)
os.makedirs('output', exist_ok=True)
os.makedirs('intermediate_splits', exist_ok=True)

print("Directories verified.")

Directories verified.


# 2.2. Data Form

* **'COFOG' Dropdown**: This dropdown lists all the unique COFOG (Classification of the Functions of Government) values found in your gfs_raw_data.xlsx file. You can select a specific spending function from this list.

* **'TYPE_OF_TRANSFORMATION' Dropdown**: This dropdown shows the unique data transformation types (e.g., 'Percent of total outlays', 'Percent of GDP') available in your data. You can choose how the data is represented.

* **'SECTOR' Dropdown**: This dropdown contains all the unique sectors (e.g., 'Central government', 'Local Government') present in your Excel file. You can filter the data by a specific sector.

* **'Data Column Name' Text Input**: This is an empty box where you can type a custom name for a new data column that will be created later in the processing pipeline.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd

# Define the path to the raw GFS data Excel file
file_path = './data/gfs_raw_data.xlsx'
# Load the data into a pandas DataFrame
df = pd.read_excel(file_path)

# Create Dropdown for COFOG values
cofog_values = df['COFOG'].unique()
dropdown_cofog = widgets.Dropdown(
    options=cofog_values,
    description='Choose COFOG:',
)
display(dropdown_cofog)

# Create Dropdown for TYPE_OF_TRANSFORMATION values
transformation_values = df['TYPE_OF_TRANSFORMATION'].unique()
dropdown_transformartion = widgets.Dropdown(
    options=transformation_values,
    description='Choose Transformation Type:',
)
display(dropdown_transformartion)

# Create Dropdown for SECTOR values
sector_values = df['SECTOR'].unique()
dropdown_sector = widgets.Dropdown(
    options=sector_values,
    description='Choose Sector:',
)
display(dropdown_sector)

# Create Text Input for new data column name
input_data_col = widgets.Text(
    placeholder='Enter new data column name here',
    description='Data Column Name:'
)
display(input_data_col)

In [ ]:
print(input_data_col.value)

## 3. Run Pipeline Stages

You can run the stages individually or use the orchestrator.

In [ ]:
# @title Orchestrator: Run Full Pipeline
# @markdown This command automates Stages 1 through 5 sequentially.

!cofog-panel run \
    --source-file ./data/gfs_raw_data.xlsx \
    --lookup-file ./data/country_codes.xlsx \
    --cofog "Defence" \
    --sector "General government" \
    --output-cols "DATA_DEFENCE"

### Individual Stages (Manual Execution)

In [ ]:
# Stage 1 & 2: Validation
!cofog-panel check-format --source-file ./data/gfs_raw_data.xlsx
!cofog-panel check-country-format --lookup-file ./data/country_codes.xlsx

In [ ]:
# Stage 3: Seeding the Master Schema
!cofog-panel seed-master --lookup-file ./data/country_codes.xlsx --output-dir ./output

In [ ]:
# Stage 4: ETL and Data Splitting
!cofog-panel split --source-file ./data/gfs_raw_data.xlsx --cofog "{dropdown_cofog.value}" --filter-type "{dropdown_transformartion.value}"

In [ ]:
# Stage 5: Harmonization and Aggregation
!cofog-panel aggregate --master-file ./output/COFOG_MASTER_SCHEMA.xlsx --folder-path ./intermediate_splits --data-col "{input_data_col.value}" --sector "{dropdown_sector.value}"